# Speculative Decoding — RL Environment Demo

This notebook runs the full speculative decoding RL environment on a **free T4 GPU** via Google Colab.

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

## What this demonstrates
- Speculative decoding achieves **2–3x speedup** over baseline on GPU
- Output token distribution is **statistically identical** to target-only generation
- The automated judge scores the implementation on **correctness + speedup**

**Algorithm:** Leviathan et al. (2023) — [Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2211.17192)

## Step 1: Setup

In [ ]:
# Clone the repo
!git clone https://github.com/Wableprajwal/speculative-decoding-rl-env.git
%cd speculative-decoding-rl-env

In [ ]:
# Install dependencies
!pip install -q transformers==4.40.0 torch numpy pytest

In [ ]:
# Verify GPU is available
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime > Change runtime type > T4 GPU')

## Step 2: Generate Eval Prompts

In [ ]:
!python data/generate_eval_prompts.py

## Step 3: Smoke Test
Quick side-by-side comparison of speculative vs baseline generation.

In [ ]:
!python scripts/smoke_test.py

## Step 4: Unit Tests
Tests correctness properties of the algorithm.

In [ ]:
!pytest tests/ -v -s 2>&1

## Step 5: Full Judge Evaluation
Runs the complete RL environment judge on all 100 eval prompts.
Checks correctness (token match rate >= 95%) and speedup (>= 1.5x on GPU).

In [ ]:
!python judge/judge.py

## Step 6: Speedup Visualisation
Visual breakdown of where time is spent.

In [ ]:
import sys, time, torch
sys.path.insert(0, '.')
from solution.speculative_decoding import speculative_decode
from solution.baseline import baseline_decode

test_prompts = [
    'The theory of relativity states that',
    'In the beginning of the universe',
    'Machine learning models are trained by',
    'The first computer was invented in',
    'Scientists recently discovered that',
]

times_base = []
times_spec = []

for prompt in test_prompts:
    t0 = time.perf_counter()
    baseline_decode(prompt=prompt, max_new_tokens=50, seed=42)
    times_base.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    speculative_decode(prompt=prompt, max_new_tokens=50, K=4, seed=42)
    times_spec.append(time.perf_counter() - t0)

import matplotlib.pyplot as plt
import numpy as np

x      = np.arange(len(test_prompts))
labels = [p[:30] + '...' for p in test_prompts]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars1 = ax1.bar(x - 0.2, times_base, 0.4, label='Baseline (target only)', color='#B4B2A9')
bars2 = ax1.bar(x + 0.2, times_spec, 0.4, label='Speculative decoding', color='#534AB7')
ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
ax1.set_ylabel('Time (seconds)')
ax1.set_title('Generation time per prompt')
ax1.legend()

# Speedup per prompt
speedups = [b / max(s, 1e-6) for b, s in zip(times_base, times_spec)]
colors   = ['#0F6E56' if s >= 1.5 else '#993C1D' for s in speedups]
ax2.bar(x, speedups, color=colors)
ax2.axhline(y=1.5, color='black', linestyle='--', linewidth=1, label='1.5x threshold')
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
ax2.set_ylabel('Speedup (x)')
ax2.set_title('Speedup vs baseline')
ax2.legend()

avg_speedup = sum(speedups) / len(speedups)
fig.suptitle(f'Speculative Decoding Results  |  Average speedup: {avg_speedup:.2f}x', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('speedup_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nAverage speedup: {avg_speedup:.2f}x')
print('Chart saved to speedup_results.png')

## Results Summary

| Metric | Value |
|--------|-------|
| Token match rate | 100% (greedy decoding — bit-for-bit identical to target-only) |
| Speedup on T4 GPU | ~2–3x |
| Judge threshold | 1.5x speedup + 95% correctness |

**Key insight:** Greedy (argmax) decoding makes acceptance deterministic: a draft token is
accepted if and only if the target model would have chosen the same token. This guarantees
the output is identical to target-only greedy generation and gives the maximum possible
token match rate (100%).

**Reference:** Leviathan, Y., Kalman, M., & Matias, Y. (2023). *Fast Inference from Transformers via Speculative Decoding.* ICML 2023.